# 🛡️ CyberSecKavach — Semantic Cybersecurity Threat Intelligence Engine

> **Author:** CyberSecKavach Project  
> **Dataset:** [Cybersecurity Threat Detection Logs](https://www.kaggle.com/datasets/aryan208/cybersecurity-threat-detection-logs) — 6,000,000 rows  
> **Model:** `all-MiniLM-L6-v2` (Sentence Transformers) + Structured Feature Hybrid  
> **Goal:** Raw logs → Semantic Understanding → Threat Detection → Severity + Explainability → Fast Inference API  

---

## 🗺️ Notebook Roadmap

| Step | Module | Description |
|------|--------|-------------|
| 1 | EDA | Explore attack patterns, distributions, and feature intelligence |
| 2 | Normalization Engine | Convert raw logs → canonical NLP sentences |
| 3 | Cleaning & Labels | Deduplicate, standardize, encode threat labels |
| 4 | Structured Features | Engineer security-aware numeric features |
| 5 | Tokenization & Embeddings | MiniLM semantic encoder pipeline |
| 6 | Threat Classifier | Train hybrid MiniLM + structured model |
| 7 | Severity Engine | SOC-style confidence + risk ranking |
| 8 | Explainability | SHAP feature attribution |
| 9 | ONNX Optimization | Fast inference export |
| 10 | Production Inference | `predict_security_log()` — deployable API function |

---

**Architecture:**
```
Raw Logs
   ↓  Normalization Engine
   ↓  MiniLM Semantic Encoder
   ↓  Hybrid Feature Vector
   ↓  Threat Classifier
   ↓  Severity + Confidence
   ↓  Explainability (SHAP)
   ↓  Inference API
```


---
## ⚙️ Step 0 — Environment Setup

Install all required libraries. Run this cell first on a fresh Kaggle/Colab session.


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess, sys

packages = [
    'sentence-transformers',
    'shap',
    'skl2onnx',
    'onnxruntime',
    'scipy',
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)

print('✅ All packages installed.')


### 📦 Core Imports


In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import os
import re
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import entropy as scipy_entropy

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='darkgrid', palette='muted')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('✅ Imports ready.')


### 📂 Load Dataset

Supports both **Kaggle** (auto-detects `/kaggle/input/`) and **Colab** (`kagglehub`) environments.


In [ ]:
# ── Load dataset — Kaggle or Colab ───────────────────────────────────────────
KAGGLE_PATH = '/kaggle/input/cybersecurity-threat-detection-logs/cybersecurity_threat_detection_logs.csv'
COLAB_DIR   = '/content/cybersecurity-threat-detection-logs'

if os.path.exists(KAGGLE_PATH):
    # Running on Kaggle
    CSV_PATH = KAGGLE_PATH
    print('📍 Environment: Kaggle')
else:
    # Running on Colab — download via kagglehub
    import kagglehub, shutil
    raw_path = kagglehub.dataset_download('aryan208/cybersecurity-threat-detection-logs')
    shutil.move(raw_path, COLAB_DIR)
    CSV_PATH = f'{COLAB_DIR}/cybersecurity_threat_detection_logs.csv'
    print('📍 Environment: Colab')

print(f'Loading: {CSV_PATH}')
df = pd.read_csv(CSV_PATH)
print(f'✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)


---
## 🔍 Step 1 — Exploratory Data Analysis

> **Goal:** Understand attack patterns, label distributions, and which fields carry security intelligence.  
> The most important question: *What makes malicious logs different from benign ones?*


In [ ]:
# ── 1a. Schema & data quality overview ──────────────────────────────────────
print('=' * 55)
print('  DATASET OVERVIEW')
print('=' * 55)
print(f'Rows:    {df.shape[0]:>12,}')
print(f'Columns: {df.shape[1]:>12}')
print()
print('Dtypes:')
print(df.dtypes.to_string())
print()
print('Null counts:')
print(df.isnull().sum().to_string())


In [ ]:
# ── 1b. Distribution analysis ────────────────────────────────────────────────
cat_cols = ['threat_label', 'protocol', 'action', 'log_type']

for col in cat_cols:
    vc = df[col].value_counts()
    pct = (vc / len(df) * 100).round(2)
    display_df = pd.DataFrame({'count': vc, 'pct%': pct})
    print(f'\n── {col.upper()} ──')
    print(display_df.to_string())


In [ ]:
# ── 1c. Visualizations ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle('CyberSecKavach — Dataset Intelligence Overview', fontsize=16, fontweight='bold', y=1.01)

# 1. Threat label
vc = df['threat_label'].value_counts()
axes[0,0].bar(vc.index, vc.values, color=sns.color_palette('Set2', len(vc)))
axes[0,0].set_title('Threat Label Distribution', fontweight='bold')
axes[0,0].set_xlabel('Threat Label')
axes[0,0].set_ylabel('Count')
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar in axes[0,0].patches:
    axes[0,0].annotate(f'{bar.get_height():,.0f}', (bar.get_x()+bar.get_width()/2, bar.get_height()),
                       ha='center', va='bottom', fontsize=9)

# 2. Protocol
vc2 = df['protocol'].value_counts()
axes[0,1].pie(vc2.values, labels=vc2.index, autopct='%1.1f%%',
              colors=sns.color_palette('pastel', len(vc2)))
axes[0,1].set_title('Protocol Distribution', fontweight='bold')

# 3. Action
vc3 = df['action'].value_counts()
axes[0,2].bar(vc3.index, vc3.values, color=['#e74c3c', '#2ecc71'])
axes[0,2].set_title('Action Distribution', fontweight='bold')
axes[0,2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# 4. Bytes transferred
sample_bytes = df['bytes_transferred'].dropna().sample(50000, random_state=42)
axes[1,0].hist(np.log1p(sample_bytes), bins=60, color='steelblue', edgecolor='white')
axes[1,0].set_title('Log(Bytes Transferred) Distribution', fontweight='bold')
axes[1,0].set_xlabel('log1p(bytes)')

# 5. Top user agents
ua_vc = df['user_agent'].value_counts().head(8)
short_labels = [u[:30]+'...' if len(u) > 30 else u for u in ua_vc.index]
axes[1,1].barh(short_labels[::-1], ua_vc.values[::-1], color=sns.color_palette('coolwarm', 8))
axes[1,1].set_title('Top 8 User Agents', fontweight='bold')
axes[1,1].set_xlabel('Count')

# 6. Log type
vc6 = df['log_type'].value_counts()
axes[1,2].bar(vc6.index, vc6.values, color=sns.color_palette('husl', len(vc6)))
axes[1,2].set_title('Log Type Distribution', fontweight='bold')
axes[1,2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA complete. Chart saved → eda_overview.png')


In [ ]:
# ── 1d. Cross-tab: threat vs action (security insight) ───────────────────────
ct = pd.crosstab(df['threat_label'], df['action'], normalize='index').round(3) * 100
print('Threat label vs Action (% of row):')
print(ct.to_string())

print('\nBytes transferred by threat label:')
print(df.groupby('threat_label')['bytes_transferred']
        .agg(['mean','median','max'])
        .applymap(lambda x: f'{x:,.1f}')
        .to_string())


---
## 🔧 Step 2 — Log Normalization Engine

> **Goal:** Convert heterogeneous raw log rows into canonical, NLP-ready cybersecurity sentences.  
> This is the most critical engineering layer — it enables semantic generalization across log sources.

**Before normalization:**
```
protocol=TCP | action=blocked | user_agent=Nmap Scripting Engine
```
**After normalization:**
```
Blocked TCP firewall request from 192.168.1.125 to 192.168.1.124 using Nmap scanner targeting /admin transferring 10889 bytes.
```
These are now semantically equivalent to MiniLM:
- *blocked TCP request* ≈ *denied TCP connection* ≈ *rejected TCP packet*


In [ ]:
# ── 2a. User-agent normalizer ─────────────────────────────────────────────────
UA_PATTERNS = [
    (r'Nmap',              'Nmap network scanner'),
    (r'sqlmap',            'sqlmap SQL injection tool'),
    (r'Nikto',             'Nikto web vulnerability scanner'),
    (r'Masscan',           'Masscan port scanner'),
    (r'ZmEu',              'ZmEu exploit scanner'),
    (r'python-requests',   'Python requests automation script'),
    (r'curl',              'curl command-line client'),
    (r'wget',              'wget download tool'),
    (r'Go-http-client',    'Go HTTP client'),
    (r'Java',              'Java HTTP client'),
    (r'libwww-perl',       'Perl LWP scanner'),
    (r'Mozilla',           None),   # handled separately below
]

def normalize_user_agent(ua: str) -> str:
    """Map verbose UA strings to compact, meaningful tokens."""
    if pd.isna(ua) or str(ua).strip() == '':
        return 'unknown agent'
    ua = str(ua).strip()
    for pattern, label in UA_PATTERNS:
        if re.search(pattern, ua, re.IGNORECASE):
            if label is not None:
                return label
            # Mozilla: extract OS hint
            match = re.search(r'\(([^)]+)\)', ua)
            os_hint = match.group(1).split(';')[0].strip() if match else 'unknown OS'
            return f'browser on {os_hint}'
    return 'other HTTP client'


# ── 2b. Core normalization function ──────────────────────────────────────────
def log_to_text(row) -> str:
    """
    Convert a raw log row (dict or pd.Series) into a canonical
    cybersecurity NLP sentence.

    Example output:
        'Blocked TCP firewall request from 192.168.1.125 to 192.168.1.124
         using Nmap network scanner targeting /admin transferring 10889 bytes.'
    """
    action   = str(row.get('action',   'unknown')).lower().strip()
    protocol = str(row.get('protocol', 'unknown')).upper().strip()
    log_type = str(row.get('log_type', 'unknown')).lower().strip()
    src_ip   = str(row.get('source_ip', 'unknown'))
    dst_ip   = str(row.get('dest_ip',   'unknown'))
    ua       = normalize_user_agent(row.get('user_agent', ''))
    path     = str(row.get('request_path', '/')).strip()
    bytes_tx = row.get('bytes_transferred', 0)

    try:
        bytes_tx = int(bytes_tx)
    except (ValueError, TypeError):
        bytes_tx = 0

    return (
        f'{action.capitalize()} {protocol} {log_type} request '
        f'from {src_ip} to {dst_ip} '
        f'using {ua} '
        f'targeting {path} '
        f'transferring {bytes_tx} bytes.'
    )


# ── 2c. Validate on sample rows ───────────────────────────────────────────────
print('=== Normalization Examples ===')
for i, (_, row) in enumerate(df.head(5).iterrows()):
    print(f'[{i}] {log_to_text(row)}')
    print(f'     → label: {row["threat_label"]}')
    print()
print('✅ Normalization engine ready.')


---
## 🧹 Step 3 — Data Cleaning & Label Engineering

> **Goal:** Remove noise, standardize categoricals, and encode threat labels for training.

| Threat | Label |
|--------|-------|
| benign | 0 |
| suspicious | 1 |
| malicious | 2 |
| port_scan | 3 |
| brute_force | 4 |
| ddos | 5 |


In [ ]:
# ── 3a. Drop nulls & duplicates ──────────────────────────────────────────────
print(f'Before cleaning: {df.shape[0]:,} rows')

df.drop_duplicates(inplace=True)
df.dropna(subset=['threat_label', 'protocol', 'action', 'bytes_transferred'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'After  cleaning: {df.shape[0]:,} rows')
print(f'Removed:         {6_000_000 - df.shape[0]:,} rows')


In [ ]:
# ── 3b. Standardize categoricals ─────────────────────────────────────────────
df['protocol']     = df['protocol'].str.upper().str.strip()
df['action']       = df['action'].str.lower().str.strip()
df['log_type']     = df['log_type'].str.lower().str.strip()
df['threat_label'] = df['threat_label'].str.lower().str.strip()
df['request_path'] = df['request_path'].fillna('/').str.strip()
df['user_agent']   = df['user_agent'].fillna('unknown')

print('Unique protocols:', sorted(df['protocol'].unique()))
print('Unique actions:  ', sorted(df['action'].unique()))
print('Unique labels:   ', sorted(df['threat_label'].unique()))


In [ ]:
# ── 3c. Label encoding ────────────────────────────────────────────────────────
LABEL_MAP = {
    'benign':      0,
    'suspicious':  1,
    'malicious':   2,
    'port_scan':   3,
    'brute_force': 4,
    'ddos':        5,
}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

df['label'] = df['threat_label'].map(LABEL_MAP)

unmapped = df['label'].isna().sum()
if unmapped > 0:
    print(f'⚠️  Unmapped labels ({unmapped} rows):')
    print(df[df['label'].isna()]['threat_label'].value_counts())
    df.dropna(subset=['label'], inplace=True)

df['label'] = df['label'].astype(int)

print('Label distribution:')
ld = df['label'].value_counts().sort_index()
for idx, cnt in ld.items():
    print(f'  {idx} ({INV_LABEL_MAP[idx]:<12}): {cnt:>10,}  ({cnt/len(df)*100:.2f}%)')

print(f'\n✅ Labels encoded. Final dataset: {df.shape[0]:,} rows')


---
## ⚙️ Step 4 — Structured Feature Engineering

> **Goal:** Add numeric security signals that complement MiniLM embeddings.  
> Transformers alone are insufficient — structured signals like byte volume, timing, and tool entropy are powerful discriminators.

| Feature | Rationale |
|---------|----------|
| `hour` | Attacks spike at off-hours |
| `day_of_week` | Weekend traffic is anomalous in enterprise |
| `bytes_log` | DDoS = high volume; recon = low volume |
| `path_depth` | Deep paths = targeted attacks |
| `ua_entropy` | High entropy → automated/scanner tools |
| `same_ip` | Loopback-style scanning indicator |
| `is_blocked` | Action signal |


In [ ]:
# ── 4a. Timestamp features ───────────────────────────────────────────────────
df['timestamp']   = pd.to_datetime(df['timestamp'], errors='coerce')
df['hour']        = df['timestamp'].dt.hour.fillna(0).astype(int)
df['day_of_week'] = df['timestamp'].dt.dayofweek.fillna(0).astype(int)

# ── 4b. Path features ────────────────────────────────────────────────────────
df['path_depth'] = df['request_path'].apply(
    lambda p: len(str(p).strip('/').split('/')) if pd.notna(p) else 0
)

# ── 4c. IP features ──────────────────────────────────────────────────────────
df['same_ip'] = (df['source_ip'] == df['dest_ip']).astype(int)

# ── 4d. User-agent entropy ───────────────────────────────────────────────────
def ua_entropy(ua: str) -> float:
    if pd.isna(ua) or len(str(ua)) == 0:
        return 0.0
    ua = str(ua)
    probs = [ua.count(c) / len(ua) for c in set(ua)]
    return float(scipy_entropy(probs))

df['ua_entropy'] = df['user_agent'].apply(ua_entropy)

# ── 4e. Byte volume ───────────────────────────────────────────────────────────
df['bytes_log'] = np.log1p(df['bytes_transferred'].fillna(0))

# ── 4f. Action binary ─────────────────────────────────────────────────────────
df['is_blocked'] = (df['action'] == 'blocked').astype(int)

# ── 4g. Protocol one-hot ──────────────────────────────────────────────────────
proto_dummies = pd.get_dummies(df['protocol'], prefix='proto').astype(int)
df = pd.concat([df, proto_dummies], axis=1)

STRUCTURED_COLS = (
    ['hour', 'day_of_week', 'path_depth', 'same_ip',
     'ua_entropy', 'bytes_log', 'is_blocked']
    + [c for c in df.columns if c.startswith('proto_')]
)

print(f'Structured features ({len(STRUCTURED_COLS)}): {STRUCTURED_COLS}')
print(df[STRUCTURED_COLS].describe().round(3).to_string())
print('\n✅ Feature engineering complete.')


---
## 🧠 Step 5 — Tokenization & Embedding Pipeline

> **Model:** `sentence-transformers/all-MiniLM-L6-v2`  
> **Why MiniLM?** Fast (5-10ms/batch), lightweight (80MB), strong semantic quality — ideal for Kaggle + HuggingFace deployment.

> ⚠️ **Note on scale:** 6M rows × embedding is GPU-hours of work. We train on a **200K stratified sample** — statistically representative and Colab/Kaggle safe. For production batch inference, use the ONNX pipeline from Step 9.


In [ ]:
# ── 5a. Load MiniLM ───────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer

print('Loading MiniLM-L6-v2...')
t0 = time.time()
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print(f'✅ Model loaded in {time.time()-t0:.1f}s')

# Sanity check — semantic similarity test
test_sentences = [
    'blocked TCP request using Nmap scanner',
    'denied TCP connection using Nmap scanner',
    'allowed HTTP request using browser on Windows',
]
test_embs = embedder.encode(test_sentences, normalize_embeddings=True)
sim_12 = float(np.dot(test_embs[0], test_embs[1]))
sim_13 = float(np.dot(test_embs[0], test_embs[2]))
print(f'\nSemantic similarity check:')
print(f'  blocked ≈ denied (same threat): {sim_12:.3f}  ← should be HIGH')
print(f'  blocked ≈ benign browser:       {sim_13:.3f}  ← should be LOW')


In [ ]:
# ── 5b. Stratified training sample ───────────────────────────────────────────
from sklearn.model_selection import train_test_split

TRAIN_SIZE = 200_000

# Stratified sample to preserve class ratios
train_df, _ = train_test_split(
    df,
    train_size=min(TRAIN_SIZE, len(df)),
    stratify=df['label'],
    random_state=RANDOM_STATE,
)
train_df = train_df.reset_index(drop=True)

print(f'Training sample: {len(train_df):,} rows')
print('Label distribution in sample:')
for lbl, cnt in train_df['label'].value_counts().sort_index().items():
    print(f'  {lbl} ({INV_LABEL_MAP[lbl]:<12}): {cnt:>8,}')


In [ ]:
# ── 5c. Generate log text + embed ────────────────────────────────────────────
print('Normalizing log text...')
t0 = time.time()
train_df['log_text'] = train_df.apply(log_to_text, axis=1)
print(f'  Done in {time.time()-t0:.1f}s')

print('Encoding with MiniLM (batched)...')
t0 = time.time()
embeddings = embedder.encode(
    train_df['log_text'].tolist(),
    batch_size=512,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f'  Done in {time.time()-t0:.1f}s')
print(f'  Embedding shape: {embeddings.shape}  (N × 384-dim)')
print('✅ Embeddings ready.')


---
## 🎯 Step 6 — Train Threat Classification Model

> **Architecture:** MiniLM embeddings (384-dim) + structured features → Logistic Regression (strong, fast, interpretable baseline)

> **Cybersecurity metric priority:**
> - F1-score (primary) — balance precision and recall
> - Recall (high priority) — missing attacks is dangerous
> - Precision (secondary) — false positives waste analyst time


In [ ]:
# ── 6a. Build feature matrix ─────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

struct_matrix = train_df[STRUCTURED_COLS].fillna(0).values.astype(np.float32)

scaler = StandardScaler()
struct_scaled = scaler.fit_transform(struct_matrix)

EMBED_DIM = embeddings.shape[1]  # 384
X = np.hstack([embeddings, struct_scaled]).astype(np.float32)
y = train_df['label'].values

print(f'Feature matrix: {X.shape}  ({EMBED_DIM} MiniLM + {len(STRUCTURED_COLS)} structured)')


In [ ]:
# ── 6b. Train / validation split ─────────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape[0]:,}  |  Val: {X_val.shape[0]:,}')


In [ ]:
# ── 6c. Train Logistic Regression ───────────────────────────────────────────
from sklearn.linear_model import LogisticRegression

print('Training...')
t0 = time.time()

clf = LogisticRegression(
    C=5.0,
    max_iter=1000,
    solver='lbfgs',
    multi_class='multinomial',
    class_weight='balanced',   # handles imbalanced threat classes
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
clf.fit(X_train, y_train)

print(f'✅ Trained in {time.time()-t0:.1f}s')


In [ ]:
# ── 6d. Evaluation ───────────────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report, f1_score, confusion_matrix, roc_auc_score
)

y_pred  = clf.predict(X_val)
y_proba = clf.predict_proba(X_val)

label_names = [INV_LABEL_MAP[i] for i in range(len(LABEL_MAP))]

print('=' * 60)
print('  VALIDATION REPORT')
print('=' * 60)
print(classification_report(y_val, y_pred, target_names=label_names))

macro_f1 = f1_score(y_val, y_pred, average='macro')
print(f'Macro F1:  {macro_f1:.4f}')

try:
    auc = roc_auc_score(y_val, y_proba, multi_class='ovr', average='macro')
    print(f'ROC-AUC:   {auc:.4f}')
except Exception as e:
    print(f'ROC-AUC skipped: {e}')

# Confusion matrix heatmap
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.title('Confusion Matrix — Threat Classification', fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Evaluation complete.')


In [ ]:
# ── 6e. Save model artifacts ─────────────────────────────────────────────────
import joblib

joblib.dump(clf,    'threat_classifier.pkl')
joblib.dump(scaler, 'struct_scaler.pkl')

print('Saved: threat_classifier.pkl')
print('Saved: struct_scaler.pkl')
print('✅ Model artifacts saved.')


---
## 🚨 Step 7 — Severity & Confidence Engine

> **Goal:** Transform raw class probabilities into SOC-ready intelligence with risk severity levels.
> Security teams need triage, not just predictions.

```
Threat: port_scan | Confidence: 96.3% | Severity: High
```


In [ ]:
# ── 7a. Severity rules (SOC-aligned) ─────────────────────────────────────────
# Structure: label → ordered list of (severity, min_confidence)
# First matching threshold wins.
SEVERITY_RULES = {
    'ddos':        [('Critical', 0.90), ('High', 0.70), ('Medium', 0.50), ('Low', 0.0)],
    'brute_force': [('Critical', 0.92), ('High', 0.75), ('Medium', 0.55), ('Low', 0.0)],
    'malicious':   [('Critical', 0.90), ('High', 0.70), ('Medium', 0.50), ('Low', 0.0)],
    'port_scan':   [('High',     0.85), ('Medium', 0.60), ('Low',  0.0)],
    'suspicious':  [('Medium',   0.70), ('Low',  0.0)],
    'benign':      [('Info',     0.0)],
}

SEVERITY_ORDER = {'Critical': 4, 'High': 3, 'Medium': 2, 'Low': 1, 'Info': 0}


def get_severity(label_name: str, confidence: float) -> str:
    rules = SEVERITY_RULES.get(label_name, [('Low', 0.0)])
    for sev, thresh in rules:
        if confidence >= thresh:
            return sev
    return 'Low'


def classify_with_severity(proba_vector: np.ndarray) -> dict:
    """Full prediction output with severity, confidence, and per-class scores."""
    label_idx  = int(np.argmax(proba_vector))
    confidence = float(proba_vector[label_idx])
    label_name = INV_LABEL_MAP[label_idx]
    severity   = get_severity(label_name, confidence)

    return {
        'threat':     label_name,
        'confidence': round(confidence * 100, 2),
        'severity':   severity,
        'all_scores': {
            INV_LABEL_MAP[i]: round(float(p) * 100, 2)
            for i, p in enumerate(proba_vector)
        },
    }


# ── 7b. Demo on validation samples ───────────────────────────────────────────
print('=== Severity Engine Demo (first 8 val samples) ===')
for i, proba in enumerate(y_proba[:8]):
    result = classify_with_severity(proba)
    true_label = INV_LABEL_MAP[y_val[i]]
    match = '✅' if result['threat'] == true_label else '❌'
    print(f"{match} [{i}] Predicted: {result['threat']:<12} "
          f"Conf: {result['confidence']:>6.2f}%  "
          f"Severity: {result['severity']:<8}  "
          f"True: {true_label}")

print('\n✅ Severity engine ready.')


---
## 💡 Step 8 — Explainability Layer (SHAP)

> **Goal:** Make the AI explain *why* it flagged a threat. Explainability is non-negotiable in cybersecurity — analysts must understand what triggered an alert.

> We use **SHAP** (SHapley Additive exPlanations) on structured features since they are directly interpretable (bytes, hour, path depth, etc.).


In [ ]:
# ── 8a. SHAP explainer setup ─────────────────────────────────────────────────
import shap

# Use structured feature slice only for SHAP (interpretable portion)
STRUCT_START = EMBED_DIM  # 384 — embedding dims come first

background   = X_train[:1000, STRUCT_START:]
X_val_struct = X_val[:200,    STRUCT_START:]

explainer  = shap.LinearExplainer(clf, background, feature_perturbation='interventional')
shap_values = explainer.shap_values(X_val_struct)  # list of arrays, one per class

print(f'SHAP values shape (per class): {np.array(shap_values).shape}')
print('✅ SHAP explainer ready.')


In [ ]:
# ── 8b. SHAP summary plots per threat class ───────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(22, 12))
axes = axes.flatten()

for i, class_name in INV_LABEL_MAP.items():
    plt.sca(axes[i])
    shap.summary_plot(
        shap_values[i],
        X_val_struct,
        feature_names=STRUCTURED_COLS,
        plot_type='bar',
        show=False,
        max_display=8,
    )
    axes[i].set_title(f'SHAP — {class_name}', fontweight='bold')

plt.suptitle('Feature Attribution by Threat Class (SHAP)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP plots saved → shap_summary.png')


In [ ]:
# ── 8c. Human-readable explanation for a single log ─────────────────────────
def explain_prediction(shap_vals_for_sample: np.ndarray,
                        feature_names: list,
                        top_n: int = 5) -> str:
    """Return a human-readable feature attribution explanation."""
    pairs = sorted(zip(np.abs(shap_vals_for_sample), shap_vals_for_sample, feature_names),
                   reverse=True)
    lines = ['Top contributing features:']
    for abs_val, raw_val, fname in pairs[:top_n]:
        direction = 'increases' if raw_val > 0 else 'decreases'
        lines.append(f'  • {fname}: {direction} threat score by {abs_val:.4f}')
    return '\n'.join(lines)


# Demo
sample_idx   = 0
pred_class   = int(clf.predict(X_val[sample_idx:sample_idx+1])[0])
explanation  = explain_prediction(shap_values[pred_class][sample_idx], STRUCTURED_COLS)

print(f'Sample prediction: {INV_LABEL_MAP[pred_class]}')
print(f'True label:        {INV_LABEL_MAP[y_val[sample_idx]]}')
print()
print(explanation)


---
## ⚡ Step 9 — ONNX Optimization for Fast Inference

> **Goal:** Export the model to ONNX for 5–10× faster inference — essential for production log streams.

| Format | Latency (1K samples) | Portability |
|--------|---------------------|-------------|
| sklearn pkl | ~200ms | Python only |
| ONNX | ~20–40ms | Any language |


In [ ]:
# ── 9a. Export to ONNX ───────────────────────────────────────────────────────
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

N_FEATURES   = X_train.shape[1]
initial_type = [('float_input', FloatTensorType([None, N_FEATURES]))]

onnx_model = convert_sklearn(clf, initial_types=initial_type, target_opset=17)

ONNX_PATH = 'threat_classifier.onnx'
with open(ONNX_PATH, 'wb') as f:
    f.write(onnx_model.SerializeToString())

size_mb = os.path.getsize(ONNX_PATH) / 1024 / 1024
print(f'✅ ONNX model saved: {ONNX_PATH}  ({size_mb:.2f} MB)')


In [ ]:
# ── 9b. Benchmark sklearn vs ONNX ────────────────────────────────────────────
import onnxruntime as rt

sess       = rt.InferenceSession(ONNX_PATH)
input_name = sess.get_inputs()[0].name
BENCH_N    = 1000

sample_f32 = X_val[:BENCH_N].astype(np.float32)

# Sklearn
t0 = time.time()
_ = clf.predict_proba(sample_f32)
sklearn_ms = (time.time() - t0) * 1000

# ONNX
t0 = time.time()
onnx_out = sess.run(None, {input_name: sample_f32})
onnx_ms  = (time.time() - t0) * 1000

speedup = sklearn_ms / onnx_ms if onnx_ms > 0 else float('inf')

print(f'Benchmark ({BENCH_N:,} samples):')
print(f'  sklearn: {sklearn_ms:.1f} ms')
print(f'  ONNX:    {onnx_ms:.1f} ms   ({speedup:.1f}× faster)')
print(f'  ONNX throughput: {BENCH_N / (onnx_ms/1000):,.0f} samples/sec')

# Correctness check
sklearn_preds = clf.predict(sample_f32)
onnx_preds   = onnx_out[0]
match_pct    = np.mean(sklearn_preds == onnx_preds) * 100
print(f'\nPrediction agreement: {match_pct:.2f}%  (should be ~100%)')
print('✅ ONNX optimization complete.')


---
## 🚀 Step 10 — Production Inference System

> **Goal:** A single function `predict_security_log(log_dict)` that wraps the entire pipeline — normalization → embedding → structured features → ONNX inference → severity output.
> This is what gets deployed behind a FastAPI endpoint or HuggingFace Space.


In [ ]:
# ── 10a. Full inference pipeline ─────────────────────────────────────────────
def predict_security_log(log_dict: dict,
                          use_onnx: bool = True,
                          explain:  bool = False) -> dict:
    """
    End-to-end inference for a single cybersecurity log.

    Parameters
    ----------
    log_dict : dict
        Raw log fields. Required keys:
            timestamp, source_ip, dest_ip, protocol, action,
            log_type, bytes_transferred, user_agent, request_path
    use_onnx : bool
        Use ONNX runtime for fast inference (default True).
    explain : bool
        Include SHAP-based explanation in output (default False).

    Returns
    -------
    dict with keys: threat, confidence, severity, all_scores,
                    log_text, [explanation]
    """
    row = pd.Series(log_dict)

    # 1. Normalize → text
    log_text  = log_to_text(row)

    # 2. Embed
    embedding = embedder.encode(
        [log_text], normalize_embeddings=True, convert_to_numpy=True
    )[0]

    # 3. Structured features
    tmp = pd.DataFrame([log_dict])
    tmp['timestamp']   = pd.to_datetime(tmp.get('timestamp', [None]), errors='coerce')
    tmp['hour']        = tmp['timestamp'].dt.hour.fillna(0).astype(int)
    tmp['day_of_week'] = tmp['timestamp'].dt.dayofweek.fillna(0).astype(int)
    tmp['path_depth']  = tmp['request_path'].apply(
        lambda p: len(str(p).strip('/').split('/')) if pd.notna(p) else 0
    )
    tmp['same_ip']    = (tmp['source_ip'] == tmp['dest_ip']).astype(int)
    tmp['ua_entropy'] = tmp['user_agent'].apply(ua_entropy)
    tmp['bytes_log']  = np.log1p(tmp['bytes_transferred'].fillna(0))
    tmp['is_blocked'] = (tmp['action'].str.lower() == 'blocked').astype(int)

    proto_dum = pd.get_dummies(tmp['protocol'], prefix='proto')
    tmp = pd.concat([tmp, proto_dum], axis=1)

    for col in STRUCTURED_COLS:
        if col not in tmp.columns:
            tmp[col] = 0

    struct_vec    = tmp[STRUCTURED_COLS].fillna(0).values.astype(np.float32)
    struct_scaled = scaler.transform(struct_vec)

    # 4. Combine features
    X_input = np.hstack([embedding.reshape(1, -1), struct_scaled]).astype(np.float32)

    # 5. Predict
    if use_onnx:
        onnx_out = sess.run(None, {input_name: X_input})
        proba    = onnx_out[1][0]   # index 1 = probabilities
    else:
        proba = clf.predict_proba(X_input)[0]

    # 6. Severity + confidence
    result = classify_with_severity(proba)
    result['log_text'] = log_text

    # 7. Optional explanation
    if explain:
        pred_class  = LABEL_MAP[result['threat']]
        shap_sample = explainer.shap_values(X_input[:, STRUCT_START:])
        result['explanation'] = explain_prediction(
            shap_sample[pred_class][0], STRUCTURED_COLS
        )

    return result


print('✅ predict_security_log() ready.')


In [ ]:
# ── 10b. Demo — multiple attack scenarios ─────────────────────────────────────
TEST_CASES = [
    {
        'name': 'Port Scan (Nmap)',
        'log': {
            'timestamp': '2024-05-01T02:30:00', 'source_ip': '10.0.0.99',
            'dest_ip': '192.168.1.1', 'protocol': 'TCP', 'action': 'blocked',
            'log_type': 'firewall', 'bytes_transferred': 512,
            'user_agent': 'Nmap Scripting Engine', 'request_path': '/admin',
        },
    },
    {
        'name': 'Benign Browser Request',
        'log': {
            'timestamp': '2024-05-01T10:15:00', 'source_ip': '192.168.1.55',
            'dest_ip': '10.0.0.1', 'protocol': 'HTTP', 'action': 'allowed',
            'log_type': 'application', 'bytes_transferred': 22000,
            'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'request_path': '/index.html',
        },
    },
    {
        'name': 'SQL Injection Attempt',
        'log': {
            'timestamp': '2024-05-01T03:45:00', 'source_ip': '185.220.101.5',
            'dest_ip': '192.168.1.10', 'protocol': 'HTTP', 'action': 'blocked',
            'log_type': 'application', 'bytes_transferred': 340,
            'user_agent': 'sqlmap/1.7.8', 'request_path': '/login?id=1%27OR%271%27=%271',
        },
    },
    {
        'name': 'Suspicious ICMP (loopback)',
        'log': {
            'timestamp': '2024-05-01T23:59:00', 'source_ip': '192.168.1.201',
            'dest_ip': '192.168.1.201', 'protocol': 'ICMP', 'action': 'blocked',
            'log_type': 'firewall', 'bytes_transferred': 36000,
            'user_agent': 'unknown', 'request_path': '/',
        },
    },
]

print('=' * 65)
print('  PRODUCTION INFERENCE — DEMO')
print('=' * 65)

for tc in TEST_CASES:
    t0 = time.time()
    result = predict_security_log(tc['log'], use_onnx=True, explain=True)
    latency = (time.time() - t0) * 1000

    print(f"\n📋 [{tc['name']}]")
    print(f"   Log:        {result['log_text'][:80]}...")
    print(f"   Threat:     {result['threat']}")
    print(f"   Confidence: {result['confidence']}%")
    print(f"   Severity:   {result['severity']}")
    print(f"   Latency:    {latency:.1f} ms")
    if 'explanation' in result:
        print(f"   {result['explanation'].splitlines()[0]}")
        for line in result['explanation'].splitlines()[1:3]:
            print(f"   {line}")

print('\n✅ Production inference system verified.')
